In [ ]:
:set -XDeriveFunctor
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XTupleSections
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XMultiParamTypeClasses
:set -XFunctionalDependencies
:set -XTypeOperators
:set -XInstanceSigs
:set -XLambdaCase

import Data.List (intercalate, nub, sort)
import Data.Maybe (fromMaybe)
import Control.Monad (forM_, when)
import Data.Monoid (Sum(..), Product(..))

putHeader :: String -> IO ()
putHeader s = do
  let line = replicate (length s + 4) '═'
  putStrLn line
  putStrLn $ "║ " ++ s ++ " ║"
  putStrLn line

putSection :: String -> IO ()
putSection s = do
  putStrLn ""
  putStrLn $ "  ▶ " ++ s
  putStrLn $ "  " ++ replicate (length s) '─'

putBox :: [String] -> IO ()
putBox ls = do
  let w = maximum (map length ls) + 4
  putStrLn $ "  ┌" ++ replicate w '─' ++ "┐"
  mapM_ (\l -> putStrLn $ "  │  " ++ l ++ replicate (w - length l - 2) ' ' ++ "  │") ls
  putStrLn $ "  └" ++ replicate w '─' ++ "┘"

putHeader "Сопряжения в Haskell"
putStrLn "Окружение настроено!"

# 🔗 Сопряжения в Haskell

## Универсальные конструкции через категориальные отношения

---

### 📋 Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | **Что такое сопряжение** | Интуиция, определение, биекция Hom |
| 2 | **Единица и коединица** | Естественные преобразования η, ε, треугольные тождества |
| 3 | **curry/uncurry** | Каноническое сопряжение (−×A) ⊣ (A→−) |
| 4 | **Free ⊣ Forgetful** | Свободные и забывающие функторы |
| 5 | **Δ ⊣ (×) и (+) ⊣ Δ** | Два сопряжённых у одного функтора |
| 6 | **State ⊣ Store** | Монада и комонада из сопряжения |
| 7 | **Adjunction² = Monad** | Сопряжения порождают монады |
| 8 | **Зоопарк Reader/Writer** | Сравнение левых и правых сопряжённых |
| 9 | **Законы и итоги** | Проверка законов, сводная таблица |
| 10 | **State/Store: глубоко** | Детальный разбор монады и комонады |
| 11 | **Representable функторы** | Функторы, изоморфные Hom(x,−) |
| 12 | **Limits и Colimits** | Пределы через сопряжения с Δ |
| 13 | **Расширения Кана** | Ran и Lan как универсальные конструкции |
| 14 | **Зоопарк сопряжений** | Полная сводная таблица |

> **Главная идея:** Сопряжение — это «оптимальный перевод» между двумя категориями.
> **Левый** сопряжённый *F ⊣ G* строит **наименьшую** структуру, **правый** — **наибольшую**.

---
## 1️⃣ Что такое сопряжение?

**Сопряжение (adjunction)** — одно из центральных понятий теории категорий.
Говорят, что функтор *F : C → D* является **левым сопряжённым** к *G : D → C* (пишут *F ⊣ G*), если существует **естественная биекция**:

$$\text{Hom}_D(F\,a,\, b) \cong \text{Hom}_C(a,\, G\,b)$$

### Интуиция

| Перспектива | Смысл |
|---|---|
| **Морфизмы** | Стрелка *F a → b* — то же, что стрелка *a → G b* |
| **Оптимальность** | *F a* — наименьший *D*-объект, построенный из *a* |
| **Факторизация** | Любой морфизм *a → G b* единственным образом факторизуется через единицу |
| **Haskell** | Биекция `(f a -> b) ≅ (a -> g b)` |

### Асимметрия: левый ≠ правый

Левый сопряжённый **свободен** и **конструктивен**: строит наиболее общую структуру.
Правый сопряжённый **консервативен**: сохраняет только нужное.

In [ ]:
-- Класс сопряжения
class (Functor f, Functor g) => Adjunction f g | f -> g, g -> f where
  unit         :: a -> g (f a)           -- η : Id → G∘F
  counit       :: f (g a) -> a           -- ε : F∘G → Id
  leftAdjunct  :: (f a -> b) -> a -> g b -- φ
  rightAdjunct :: (a -> g b) -> f a -> b -- φ⁻¹
  leftAdjunct  f = fmap f . unit
  rightAdjunct g = counit . fmap g

putHeader "Класс Adjunction"
putStrLn ""
putBox [ "F ⊣ G"
       , ""
       , "unit   :: a -> g (f a)     η : Id → G∘F"
       , "counit :: f (g a) -> a     ε : F∘G → Id"
       , ""
       , "leftAdjunct  : (f a -> b) -> a -> g b"
       , "rightAdjunct : (a -> g b) -> f a -> b"
       ]
putStrLn ""
putStrLn "Биекция φ и φ⁻¹ взаимно обратны:"
putStrLn "  leftAdjunct  (rightAdjunct f) = f"
putStrLn "  rightAdjunct (leftAdjunct  g) = g"

---
## 2️⃣ Единица и коединица

Сопряжение *F ⊣ G* полностью определяется парой **естественных преобразований**:

$$\eta_a : a \to G(F(a)) \qquad \text{(единица, unit)}$$
$$\varepsilon_b : F(G(b)) \to b \qquad \text{(коединица, counit)}$$

### Треугольные тождества

$$\varepsilon_{F(a)} \circ F(\eta_a) = \mathrm{id}_{F(a)}$$
$$G(\varepsilon_b) \circ \eta_{G(b)} = \mathrm{id}_{G(b)}$$

### Схема

![Треугольные тождества](../diagrams/adjunctions/adj_triangle.svg)

### Связь с биекцией

```haskell
leftAdjunct  f = fmap f . unit    -- поднять через G
rightAdjunct g = counit . fmap g  -- опустить через F

-- Из биекции получаем unit/counit:
unit   = leftAdjunct  id
counit = rightAdjunct id
```

In [ ]:
-- Треугольные тождества в виде типов
triangleLaw1 :: (Adjunction f g) => f a -> f a
triangleLaw1 = rightAdjunct unit   -- = id

triangleLaw2 :: (Adjunction f g) => g a -> g a
triangleLaw2 = leftAdjunct counit  -- = id

-- Иллюстрация на curry/uncurry
-- Сопряжение (b, -) -| (b ->-): unit b = \a -> (b, a), counit (f, a) = f a

ex_triangle1 :: (Int, Char) -> (Int, Char)
ex_triangle1 = uncurry (\b -> \a -> (b, a))  -- = id

-- Для второго тождества используем подходящую функцию типа Char -> Int
import Data.Char (ord)

ex_triangle2 :: (Char -> Int) -> (Char -> Int)
ex_triangle2 f = fst . (\a -> (f a, a))    -- = id = f

putHeader "Треугольные тождества"

putSection "Тождество 1: uncurry (\\b a -> (b, a)) = id"
print $ ex_triangle1 (42, 'x')   -- (42, 'x')
print $ ex_triangle1 (0,  'z')   -- (0,  'z')

putSection "Тождество 2: fst . (\\a -> (f a, a)) = f"
let f = ord  -- ord :: Char -> Int
print $ ex_triangle2 f 'A'   -- 65 = ord 'A'
print $ ex_triangle2 f 'z'   -- 122 = ord 'z'
print $ f 'A' == ex_triangle2 f 'A'  -- True

putSection "Интерпретация"
putStrLn "Если добавить структуру (F) а затем сразу забыть (e) — получим исходное."
putStrLn "Если вложить (n) а затем сразу забыть (G*e) — тоже."

---
## 3️⃣ Каноническое сопряжение: curry / uncurry

$$(- \times A) \dashv (A \to -)$$

Функтор «умножить на *A*» есть **левый сопряжённый** к функтору «функции из *A*»:

$$\text{Hom}(B \times A,\, C) \cong \text{Hom}(B,\, A \to C)$$

В Haskell это буквально `curry` и `uncurry`!

### Компоненты

| Компонент | Тип | Реализация |
|---|---|---|
| Единица *η_b* | `b -> a -> (b, a)` | `\b a -> (b, a)` |
| Коединица *ε* | `(a -> c, a) -> c` | `\(f, a) -> f a` |
| *leftAdjunct* | `((b,a) -> c) -> b -> a -> c` | `curry` |
| *rightAdjunct* | `(b -> a -> c) -> (b,a) -> c` | `uncurry` |

> Это не просто синтаксический сахар. Это **универсальное свойство произведения** в **Hask**.

In [ ]:
-- Сопряжение (−,A) ⊣ (A→−)
newtype PairWith a b = PairWith { unPair :: (b, a) } deriving (Show)
newtype FunFrom  a b = FunFrom  { unFun  :: a -> b  }

instance Functor (PairWith a) where
  fmap f (PairWith (b, a)) = PairWith (f b, a)

instance Functor (FunFrom a) where
  fmap f (FunFrom g) = FunFrom (f . g)

instance Adjunction (PairWith a) (FunFrom a) where
  unit b           = FunFrom $ \a -> PairWith (b, a)
  counit (PairWith (FunFrom f, a)) = f a
  leftAdjunct  f b = FunFrom $ \a -> f (PairWith (b, a))
  rightAdjunct g (PairWith (b, a)) = unFun (g b) a

putHeader "curry/uncurry как сопряжение (−×A) ⊣ (A→−)"

putSection "Единица η_b = \\a -> (b, a)"
let eta5 = \a -> (5 :: Int, a :: Char)
print $ eta5 'x'   -- (5, 'x')
print $ eta5 'z'   -- (5, 'z')

putSection "Коединица ε_(f, a) = f a"
let eps (f, a) = f a :: Int
print $ eps ((*2),   21)   -- 42
print $ eps ((+100),  7)   -- 107

putSection "Биекция curry ↔ uncurry"
let f = \(x, y) -> x * y + 1 :: Int
let g  = curry f    -- leftAdjunct : (B,A)->C becomes B->A->C
let f' = uncurry g  -- rightAdjunct: inverse
print $ f  (3, 4)   -- 13
print $ g   3  4    -- 13
print $ f' (3, 4)   -- 13 = f

putBox [ "leftAdjunct  = curry"
       , "rightAdjunct = uncurry"
       , ""
       , "curry (uncurry f) = f"
       , "uncurry (curry f) = f"
       ]

---
## 4️⃣ Free ⊣ Forgetful: свободные и забывающие функторы

Паттерн **Free/Forgetful** — архетипный пример сопряжений в алгебре.

$$\text{Hom}_{\text{Alg}}(F(a),\, m) \cong \text{Hom}_{\text{Set}}(a,\, U(m))$$

> **Задать гомоморфизм из свободного объекта** = **задать функцию на порождающих**

### Примеры

| Алгебра | Свободный объект | В Haskell |
|---|---|---|
| Моноид | Список `[a]` | `foldMap` |
| Монада | `Free f a` | `foldFree` |
| Полугруппа | `NonEmpty a` | `foldMap1` |

### В Haskell: `foldMap` = rightAdjunct

Единица *η : a → [a]* = `pure`.  
Коединица *ε : [[a]] → [a]* = `concat` (= `join`).  
`foldMap` = `rightAdjunct` для сопряжения `[] ⊣ U`.

In [ ]:
-- Свободный моноид: [] ⊣ Monoid
putHeader "Свободный моноид: [] как Free Monoid"

putSection "η : a → [a] (= pure)"
let eta = pure :: Int -> [Int]
print $ eta 42    -- [42]

putSection "ε : [[a]] → [a] (= concat = join)"
print $ concat [[1,2],[3,4],[5 :: Int]]  -- [1,2,3,4,5]

putSection "rightAdjunct = foldMap: гомоморфизм из [a]"
print $ foldMap Sum     [1..5 :: Int]    -- Sum {getSum = 15}
print $ foldMap Product [1..5 :: Int]    -- Product {getProduct = 120}
print $ foldMap show    [1..5 :: Int]    -- "12345"

putSection "Биекция: функция a->m ⇔ гомоморфизм [a]->m"
let f = show :: Int -> String
let lifted = foldMap f     -- rightAdjunct f
print $ lifted [1,2,3]     -- "123"
let back x = lifted [x]    -- leftAdjunct lifted = f
print $ back 42            -- "42" = f 42

putStrLn ""
putBox [ "Задать гомоморфизм из [a] в m"
       , "= задать функцию a -> m"
       , ""
       , "foldMap f = rightAdjunct f"
       ]

---
## 5️⃣ Два сопряжённых у одного функтора: Диагональный функтор

Диагональный функтор **Δ : C → C×C** отправляет *a* в *(a, a)*.
У него есть **два сопряжённых**:

$$+ \dashv \Delta \dashv \times$$

| Сопряжение | Смысл | В Haskell |
|---|---|---|
| $(+) \dashv \Delta$ | Either — **левый** | `either f f :: Either a a -> b` |
| $\Delta \dashv (\times)$ | Произведение — **правый** | `\x -> (x,x) :: a -> (a,a)` |

### Левое (+) ⊣ Δ

$$\text{Hom}(A + A,\, B) \cong \text{Hom}(A, B) \times \text{Hom}(A, B)$$

Задать функцию из `Either a a` = задать **две** функции из `a`.

### Правое Δ ⊣ (×)

$$\text{Hom}(A, B_1 \times B_2) \cong \text{Hom}(A, B_1) \times \text{Hom}(A, B_2)$$

Задать функцию в произведение = задать **пару** функций.

In [ ]:
putHeader "Диагональный функтор: + ⊣ Δ ⊣ ×"

-- Диагональный функтор
delta :: a -> (a, a)
delta x = (x, x)

-- ПРАВОЕ: Δ ⊣ (×)
-- rightAdjunct: (a->b, a->c) → (a -> (b,c))
pairAdj :: (a -> b) -> (a -> c) -> a -> (b, c)
pairAdj f g x = (f x, g x)
-- leftAdjunct: (a->(b,c)) → (a->b, a->c)
unpairAdj :: (a -> (b, c)) -> (a -> b, a -> c)
unpairAdj h = (fst . h, snd . h)

-- ЛЕВОЕ: (+) ⊣ Δ
-- rightAdjunct: (a->b, a->b) → (Either a a -> b)
eitherAdj :: (a -> b) -> (a -> b) -> Either a a -> b
eitherAdj f _ (Left  x) = f x
eitherAdj _ g (Right x) = g x

putSection "Правое сопряжение Δ ⊣ (×)"
let encode :: Int -> (String, Bool)
    encode = pairAdj show even
print $ encode 42   -- ("42", True)
print $ encode 7    -- ("7",  False)
print $ delta 5     -- (5, 5)
putStrLn $ "  η (delta) 5 = " ++ show (delta (5 :: Int))
putStrLn "  ε (f, g) = применить обе функции"

putSection "Левое сопряжение (+) ⊣ Δ"
let handle :: Either Int Int -> Int
    handle = eitherAdj negate abs
print $ handle (Left   5)    -- -5   (negate)
print $ handle (Right (-3))  --  3   (abs)
putStrLn $ "  η a = Left a (or Right a)"
putStrLn $ "  ε (Left a)  = a  (either id id)"
putStrLn $ "  ε (Right a) = a"

putSection "Сравнение левого и правого"
putBox [ "ЛЕВОЕ  (+) ⊣ Δ:       Either a a = копроизведение"
       , "  η : a → Either a a    ε : (a, a) → a  (either id id)"
       , ""
       , "ПРАВОЕ Δ ⊣ (×):         (a, a) = произведение"
       , "  η : a → (a, a)         ε : (f, g) → pairAdj f g"
       ]

---
## 6️⃣ State ⊣ Store: монада и комонада из одного сопряжения

Сопряжение $(- \times S) \dashv (S \to -)$ порождает две двойственные структуры:

```
State s a = s → (a, s)     -- монада  (G ∘ F)
Store s a = (s → a, s)     -- комонада (F ∘ G)
```

### Компоненты

| Компонент | State | Store |
|---|---|---|
| Единица *η* | `return :: a -> State s a` | `pos :: Store s s` |
| Коединица *ε* | `runState` | `extract` |
| Связь | `join = G(ε_F)` | `duplicate = F(η_G)` |

### Монада = G ∘ F, комонада = F ∘ G

Монада State и комонада Store являются **двойственными структурами** одного сопряжения.

In [ ]:
-- Монада State
newtype State s a = State { runState :: s -> (a, s) }

instance Functor (State s) where
  fmap f (State g) = State $ \s -> let (a, s') = g s in (f a, s')

instance Applicative (State s) where
  pure a = State $ \s -> (a, s)
  State f <*> State x = State $ \s ->
    let (g, s')  = f s
        (a, s'') = x s'
    in (g a, s'')

instance Monad (State s) where
  return = pure
  State x >>= f = State $ \s ->
    let (a, s') = x s
    in runState (f a) s'

get :: State s s
get = State $ \s -> (s, s)

put :: s -> State s ()
put s = State $ \_ -> ((), s)

modify :: (s -> s) -> State s ()
modify f = State $ \s -> ((), f s)

-- Комонада Store
newtype Store s a = Store { runStore :: (s -> a, s) }

instance Functor (Store s) where
  fmap f (Store (g, s)) = Store (f . g, s)

extract :: Store s a -> a
extract (Store (f, s)) = f s

duplicate :: Store s a -> Store s (Store s a)
duplicate (Store (f, s)) = Store (\s' -> Store (f, s'), s)

putHeader "State и Store: двойственные структуры сопряжения"

putSection "Монада State: s -> (a, s)"
let counter :: State Int [Int]
    counter = do
      n <- get
      put (n + 1)
      m <- get
      put (m * 2)
      k <- get
      return [n, m, k]
print $ runState counter 5   -- ([5,6,12], 12)

putSection "Единица η = return"
print $ runState (return 42 :: State Int Int) 100   -- (42, 100)

putSection "Комонада Store: (s -> a, s)"
let store = Store ((*2), 21 :: Int) :: Store Int Int
putStrLn $ "extract (Store (*2, 21)) = " ++ show (extract store)  -- 42

putSection "extract . return = id (треугольное тождество)"
print $ extract (Store (id, 42 :: Int))  -- 42

putBox [ "State s a = s -> (a, s)   -- монада  G ∘ F"
       , "Store s a = (s -> a, s)   -- комонада F ∘ G"
       , ""
       , "Двойственные структуры одного сопряжения!"
       ]

---
## 7️⃣ Сопряжения порождают монады: Monad = Adjunction²

**Теорема**: Любое сопряжение *F ⊣ G* порождает монаду *G ∘ F* и комонаду *F ∘ G*.

$$\text{return} = \eta : a \to G(F(a))$$
$$\text{join} = G(\varepsilon_F) : G(F(G(F(a)))) \to G(F(a))$$

### Известные монады из сопряжений

| Сопряжение | Монада G∘F | Комонада F∘G |
|---|---|---|
| $(-\times S) \dashv (S\to-)$ | State s | Store s |
| $\Sigma \dashv \Pi$ | Reader | Env |
| Writer ⊣ Id | Writer w | Traced w |
| [] ⊣ U | Монада списка | |

### Клейсли и алгебры Мора

Каждая монада порождается сопряжением в категории **Kleisli** (или **Eilenberg-Moore**).
Но не каждая монада порождается сопряжением в категории **Hask**.

In [ ]:
-- Монада из сопряжения F ⊣ G:
-- return = unit = η
-- join   = fmap counit  (= G(ε_F))
-- (>>=) = \ma f -> join (fmap f ma)

putHeader "Монада = G ∘ F (композиция сопряжённых)"

putSection "return = unit = η"
print $ (return 42 :: [Int])      -- [42]
print $ (return 42 :: Maybe Int)  -- Just 42

putSection "join = concat (= G(ε_F) для списка)"
print $ concat [[1,2],[3,4],[5 :: Int]]  -- [1,2,3,4,5]

putSection "(>>=) = композиция Kleisli"
let safeDiv x 0 = Nothing
    safeDiv x y = Just (x `div` y)
print $ Just 100 >>= safeDiv 10   -- Just 10
print $ Just 0   >>= safeDiv 10   -- Nothing
print $ Nothing  >>= safeDiv 10   -- Nothing

putSection "Цепочка do-нотации = Kleisli-композиция"
let process :: Int -> Maybe Int
    process x = do
      a <- safeDiv 100 x
      b <- safeDiv 50  a
      return (b + 1)
print $ process 5    -- Just 11  (100/5=20, 50/20=2, 2+1=3) — нет ошибки
print $ process 0    -- Nothing
print $ process 2    -- Just (50/50 + 1) = Just 2

putBox [ "Компоненты монады через сопряжение:"
       , ""
       , "return = unit η"
       , "join   = fmap counit . unit  (G(ε_F))"
       , "(>>=)  = rightAdjunct . leftAdjunct"
       ]

---
## 8️⃣ Сравнительный зоопарк: левые vs правые

### Reader и Writer: один функтор — два сопряжённых

Функтор `(->) r` (функции из `r`) участвует в сопряжении:

$$((,)\, r) \dashv ((->) r)$$

| Роль | Функтор | Тип | Монада |
|---|---|---|---|
| **Левый** | `(,) r` = умножить на r | `(r, a)` | Writer r |
| **Правый** | `(->) r` = функции из r | `r -> a` | Reader r |

Сопряжение $((,)\, r) \dashv ((->) r)$ — это то же самое, что curry/uncurry!

$$\text{Hom}(r \times a,\, b) \cong \text{Hom}(a,\, r \to b)$$

### Reader vs Writer

| Аспект | Reader r a | Writer r a |
|---|---|---|
| Тип | `r -> a` | `(a, r)` |
| Роль | читает окружение | пишет лог |
| Функтор | `(->) r` (= правый) | `(,) r` (= левый) |
| Единица | `\a _ -> a` (игнорирует r) | `(a, mempty)` |
| Коединица | применить к r | взять значение |

In [ ]:
putHeader "Сравнение: Reader vs Writer (((,)r) ⊣ ((->) r))"

-- Reader = (->) r   (правый сопряжённый)
type Reader r a = r -> a

ask :: Reader r r
ask = id

runReader :: Reader r a -> r -> a
runReader = id

localR :: (r -> r) -> Reader r a -> Reader r a
localR f m = m . f

-- Writer = (,) r   (левый сопряжённый)
type Writer r a = (a, r)

tell :: r -> Writer r ()
tell w = ((), w)

runWriter :: Writer r a -> (a, r)
runWriter = id

putSection "Reader: r -> a (= (->) r = правый сопряжённый)"
let greet :: Reader String String
    greet = \name -> "Hello, " ++ name ++ "!"
print $ runReader greet "World"     -- "Hello, World!"
print $ runReader greet "Haskell"   -- "Hello, Haskell!"
print $ runReader (localR (++ "!") greet) "Cat"  -- "Hello, Cat!!"

putSection "Writer: (a, r) (= (,) r = левый сопряжённый)"
let (val, log) = (42 :: Int, "computed 42" :: String)
print val  -- 42
print log  -- "computed 42"

-- Сопряжение: биекция = curry/uncurry
putSection "Биекция: ((,)r) ⊣ ((->) r) = curry/uncurry!"
let f :: (String, Int) -> String
    f (env, x) = env ++ show x
print $ f ("val=", 42)          -- "val=42"
print $ curry f "val=" 42       -- "val=42"  (leftAdjunct)
print $ uncurry (curry f) ("val=", 42)  -- "val=42"  (rightAdjunct)

putBox [ "Reader r a = r -> a      -- правый, (->) r"
       , "Writer r a = (a, r)      -- левый,  (,)  r"
       , ""
       , "Биекция ((,)r) ⊣ (->)r = curry/uncurry!"
       ]

---
## 9️⃣ Законы и итоги

### Лемма: биекция leftAdjunct/rightAdjunct

Для любого сопряжения *F ⊣ G*:

$$\text{rightAdjunct}(\text{leftAdjunct}(f)) = f$$
$$\text{leftAdjunct}(\text{rightAdjunct}(g)) = g$$

### Ключевые связи

| Понятие | Связь с сопряжениями |
|---|---|
| Монада | G ∘ F для F ⊣ G |
| Комонада | F ∘ G для F ⊣ G |
| Произведение | правое сопряжение для диагонали |
| Копроизведение | левое сопряжение для диагонали |
| Карринг | сопряжение (−×A) ⊣ (A→−) |
| foldMap | rightAdjunct для [] ⊣ U |
| State/Store | G∘F / F∘G для (−×S) ⊣ (S→−) |

### Цитата

> «Сопряжения — самый важный концепт в математике.» (Saunders Mac Lane)

> Каждый раз, когда вы видите `curry`/`uncurry`, `foldMap`, `State`, `Reader` — вы видите сопряжения.

In [ ]:
-- Проверка законов
putHeader "Проверка законов сопряжения"

-- Закон 1: rightAdjunct . leftAdjunct = id  (для curry/uncurry)
putSection "Закон 1: rightAdjunct . leftAdjunct = id"
let f0 = \(x, y) -> x + y :: Int
let f0' = uncurry (curry f0)
print $ all (\p -> f0 p == f0' p) [(x,y) | x <- [1..5 :: Int], y <- [1..5]]  -- True

-- Закон 2: leftAdjunct . rightAdjunct = id
putSection "Закон 2: leftAdjunct . rightAdjunct = id"
let g0 = \x y -> x * y :: Int
let g0' = curry (uncurry g0)
print $ all (\x -> all (\y -> g0 x y == g0' x y) [1..5 :: Int]) [1..5]  -- True

-- Треугольные тождества для foldMap
putSection "Треугольные тождества foldMap"
-- foldMap (:[]) xs = xs   (ε ∘ F(η) = id)
print $ foldMap (:[]) [1,2,3 :: Int]  -- [1,2,3]
-- \x -> foldMap f [x] = f x   (G(ε) ∘ η = id)
let h = show :: Int -> String
print $ all (\x -> foldMap h [x] == h x) [1..10 :: Int]  -- True

-- Итоговый обзор
putStrLn ""
putBox [ "Резюме сопряжений в Haskell:"
       , ""
       , "  (−×A) ⊣ (A→−)  ←→  curry / uncurry"
       , "  [] ⊣ U         ←→  pure / foldMap"
       , "  (+) ⊣ Δ ⊣ (×)  ←→  either / (\\x->(x,x))"
       , "  F ⊣ G (State)  ←→  return / join монады"
       , "  ((,)r) ⊣ (->)r ←→  curry = Reader/Writer"
       , ""
       , "  Левый  = свободный, наименьший (free)"
       , "  Правый = забывающий, наибольший (forgetful)"
       ]

---
## 1️⃣0️⃣ State Monad / Store Comonad: глубокое погружение ⊣ Store: монада и комонада из одного сопряжения

Это один из самых красивых примеров: **одно сопряжение порождает сразу монаду и комонаду**.

Сопряжение: $(-\times S) \dashv (S \to -)$

| Роль | Тип |
|------|-----|
| Левый сопряжённый F | `(-, s)` = `(a, s)` |
| Правый сопряжённый G | `(s ->)` = `s -> a` |
| Монада G∘F | `s -> (a, s)` = **State s a** |
| Комонада F∘G | `(s -> a, s)` = **Store s a** |

Ключевая симметрия: монада и комонада — **зеркальные отражения** друг друга через сопряжение.

In [ ]:
-- State как G∘F (правый ∘ левый)
-- G(F(a)) = (s ->) ((a, s)) = s -> (a, s)
newtype State' s a = State' { runState' :: s -> (a, s) }

instance Functor (State' s) where
  fmap f (State' g) = State' $ \s -> let (a, s') = g s in (f a, s')

instance Applicative (State' s) where
  pure a = State' $ \s -> (a, s)           -- unit = pure
  State' f <*> State' x = State' $ \s ->
    let (g, s')  = f s
        (a, s'') = x s'
    in (g a, s'')

instance Monad (State' s) where
  return = pure
  State' m >>= f = State' $ \s ->         -- bind через сопряжение
    let (a, s') = m s
        State' m' = f a
    in m' s'

-- Store как F∘G (левый ∘ правый)
-- F(G(a)) = ((s -> a), s)
data Store' s a = Store' (s -> a) s

instance Functor (Store' s) where
  fmap f (Store' g s) = Store' (f . g) s

-- Комонадические операции
extract' :: Store' s a -> a
extract' (Store' f s) = f s   -- counit

duplicate' :: Store' s a -> Store' s (Store' s a)
duplicate' (Store' f s) = Store' (Store' f) s  -- delta через сопряжение

-- Демонстрация симметрии
putHeader "State ⊣ Store: симметрия монады и комонады"

putSection "State: get/put через монаду"
let getS = State' $ \s -> (s, s)
    putS s = State' $ \_ -> ((), s)
    prog = do { s <- getS; putS (s + 10); return s }
print $ runState' prog 42    -- (42, 52): вернули старое, записали новое

putSection "Store: peek/seek через комонаду"
let store = Store' (*2) 5       -- "фокус" на 5, значение = 5*2 = 10
print $ extract' store           -- 10
print $ extract' (fmap (+1) store)  -- 11
let Store' f s = duplicate' store
print $ extract' (f 7)           -- 14 = 7*2

---
## 1️⃣1️⃣ Representable функторы и сопряжения

**Representable функтор** — это функтор F, изоморфный `(−→x)` для некоторого объекта x:

$$F \cong \text{Hom}(x, -)$$

Это означает, что F является **правым сопряжённым** к `Hom(x,-)`.

В Haskell: `Representable f` если `f a ≅ (Rep f -> a)`

```
Rep ((->) r) = r      -- функции сами representable!
Rep Stream   = Nat    -- Stream a ≅ Nat -> a
Rep (Store s) = s     -- Store s a ≅ s -> a
```

**Связь с сопряжением:** F representable ⟺ существует сопряжение `Δ ⊣ F` (диагональ ⊣ F).

In [ ]:
-- (implementing our own Representable for Stream below)

-- Стрим как Representable: Stream a ≅ Nat -> a
-- (используем ZipList как упрощение)
import Data.List (unfoldr)

-- Nat как тип индексов
type Nat = Int

-- Stream через функцию
newtype Stream a = Stream { runStream :: Nat -> a }

instance Functor Stream where
  fmap f (Stream g) = Stream (f . g)

-- tabulate и index -- двусторонний изоморфизм
tabulateS :: (Nat -> a) -> Stream a
tabulateS = Stream

indexS :: Stream a -> Nat -> a
indexS = runStream

-- Примеры потоков
nats :: Stream Int
nats = tabulateS id          -- 0, 1, 2, 3, ...

-- Fast fibs via zipWith trick (memoized via laziness)
fibList :: [Integer]
fibList = 0 : 1 : zipWith (+) fibList (tail fibList)

fibs :: Stream Integer
fibs = tabulateS (fibList !!)

putHeader "Representable: Stream ≅ (Nat -> a)"

putSection "Потоки через функции"
let take10 s = map (indexS s) [0..9]
print $ take10 nats           -- [0..9]
print $ take 10 fibList  -- первые 10 фибоначчи

putSection "fmap = композиция"
print $ take10 (fmap (*2) nats)  -- [0,2,4,6,8,10,12,14,16,18]

putSection "Representable: tabulate . index = id"
let doubled = fmap (*2) nats
let roundtrip = tabulateS (indexS doubled)  -- tabulate . index
print $ all (\i -> indexS doubled i == indexS roundtrip i) [0..100]  -- True

---
## 1️⃣2️⃣ Limits и Colimits через сопряжения

Пределы и копределы можно описать через сопряжения с **диагональным функтором** Δ.

**Диагональный функтор** Δ: C → C^J отправляет объект A в константный функтор, равный A везде.

$$\text{Limit} \dashv \Delta \dashv \text{Colimit}$$

| Конструкция | Сопряжение | Haskell |
|-------------|-----------|---------|
| Произведение | Δ ⊣ (×) | `(a,b)` |
| Копроизведение | (+) ⊣ Δ | `Either a b` |
| Терминальный объект | Δ ⊣ () | `()` |
| Начальный объект | Void ⊣ Δ | `Void` |
| Равнитель | Δ ⊣ Eq | функции типов |

Это объясняет **универсальные свойства**: произведение — это именно то, что нужно для сопряжения с Δ.

In [ ]:
import Data.Void (Void, absurd)

-- Произведение как правый предел (правый сопряжённый к Δ)
-- Δ ⊣ (×): Hom(Δa, (b,c)) ≅ Hom(a, b×c)

-- Универсальное свойство произведения:
fanout :: (a -> b) -> (a -> c) -> a -> (b, c)
fanout f g x = (f x, g x)    -- единственная стрелка a -> b×c

fst' :: (a, b) -> a
fst' = fst                    -- проекции -- коединица

snd' :: (a, b) -> b
snd' = snd

-- Копроизведение как левый предел (левый сопряжённый к Δ)
-- (+) ⊣ Δ: Hom(a+b, Δc) ≅ Hom(a,c) × Hom(b,c)

fanin :: (a -> c) -> (b -> c) -> Either a b -> c
fanin f _ (Left a)  = f a    -- единственная стрелка a+b -> c
fanin _ g (Right b) = g b

inl :: a -> Either a b
inl = Left                    -- инъекции -- единица

inr :: b -> Either a b
inr = Right

-- Начальный объект: Void ⊣ Δ
fromVoid :: Void -> a
fromVoid = absurd             -- из пустоты всё что угодно

-- Проверка универсальных свойств
putHeader "Limits и Colimits как сопряжения"

putSection "Произведение: Δ ⊣ (×)"
let pair = fanout (+1) (*2) 5   -- единственная стрелка
print pair                       -- (6, 10)
print $ fst pair                 -- 6 = проекция
print $ snd pair                 -- 10 = проекция

putSection "Копроизведение: (+) ⊣ Δ"
let summate = fanin (+100) (*100)
print $ summate (Left 5)    -- 105
print $ summate (Right 5)   -- 500

putSection "Начальный объект"
-- fromVoid :: Void -> Int  -- нет значений типа Void, поэтому функция тривиальна
putStrLn "Void -> a: функция определена, но никогда не вызовется"
putStrLn "Терминальный объект () -> a: единственное значение ()"
print (() :: ())

---
## 1️⃣3️⃣ Расширения Кана: всё есть расширение Кана

> «Концепция расширений Кана — возможно, самая важная концепция теории категорий.» — Saunders Mac Lane

**Расширения Кана** (Kan extensions) — способ "продолжить" функтор вдоль другого:

Дано: F: C → D, K: C → C'
Найти: LanK F: C' → D (левое) и RanK F: C' → D (правое)

$$\text{Lan}_K F \dashv (- \circ K) \dashv \text{Ran}_K F$$

В Haskell: если K = Yoneda, то получаем **плотность**.

```
Lan K F c' = ∃c. (K c -> c', F c)    -- левое: экзистенциальный тип
Ran K F c' = ∀c. (c' -> K c) -> F c  -- правое: универсальный тип (Codensity)
```

**Факт:** любой функтор — это расширение Кана вдоль тождественного функтора.

In [ ]:
-- Правое расширение Кана (Ran) вдоль тождественного функтора
-- Ran Id F ≅ F   (любой функтор = своё собственное расширение)

-- Codensity монада = Ran T T для монады T
-- Это мощный способ улучшить производительность монады

newtype Codensity m a = Codensity { runCodensity :: forall b. (a -> m b) -> m b }

instance Functor (Codensity m) where
  fmap f (Codensity c) = Codensity $ \k -> c (k . f)

instance Applicative (Codensity m) where
  pure a = Codensity $ \k -> k a
  Codensity f <*> Codensity x = Codensity $ \k ->
    f $ \g -> x $ \a -> k (g a)

instance Monad (Codensity m) where
  return = pure
  Codensity m >>= f = Codensity $ \k ->
    m $ \a -> runCodensity (f a) k

-- Лифт из базовой монады
liftCodensity :: Monad m => m a -> Codensity m a
liftCodensity m = Codensity (m >>=)

-- Опускание обратно
lowerCodensity :: Monad m => Codensity m a -> m a
lowerCodensity (Codensity f) = f return

putHeader "Расширения Кана: Codensity монада"

putSection "Codensity: улучшение производительности"
-- Проблема: ([] >>= f) >>= g медленно для списков (левоассоциативно)
-- Codensity делает это O(n) вместо O(n²)

let slowList = foldl (>>=) (return 1 :: [Int]) (replicate 5 (\x -> [x, x+1]))
let fastList = lowerCodensity $ foldl (>>=) (liftCodensity [1 :: Int])
                                 (replicate 5 (\x -> liftCodensity [x, x+1]))

print $ length slowList    -- 2^100 ... только первый элемент быстро
print $ head fastList      -- 1 (то же самое, но быстрее через CPS)

putSection "Ран тождественного: Ran Id F ≅ F"
-- lowerCodensity . liftCodensity = id (для любой монады)
let xs = [1..5 :: Int]
print $ lowerCodensity (liftCodensity xs)  -- [1,2,3,4,5]

---
## 1️⃣4️⃣ Зоопарк сопряжений в Haskell

Полная сводная таблица всех сопряжений из этого ноутбука и смежных:

| Левый F | Правый G | Монада G∘F | Комонада F∘G | Где используется |
|---------|---------|-----------|-------------|-----------------|
| `(−×A)` | `(A→−)` | `Reader A` | `(−,A)` / Writer | Карринг |
| `Free f` | `Forget` | `Free f` | нет | Свободные структуры |
| `(+)` / Either | `Δ` (диаг.) | `Either`/ExceptT | нет | Копроизведение |
| `Δ` (диаг.) | `(×)` / pair | `(,)` | `(,)` | Произведение |
| `(−×S)` | `(S→−)` | `State S` | `Store S` | State/Store |
| `LanK` | `RanK` | Codensity | Density | Оптимизация |
| `Σ` | `Π` | Dependent State | нет | Зависимые типы |
| `[]` | `U` (forgetful) | Свободный моноид | нет | Списки |

**Ключевые наблюдения:**
- Каждая монада возникает из сопряжения (теорема Клайсли)
- Правый сопряжённый сохраняет пределы, левый — копределы
- Сопряжение ↔ биекция морфизмов ↔ универсальное свойство

In [ ]:
-- Проверка ключевых сопряжений программно
putHeader "Зоопарк сопряжений: финальная проверка"

putSection "1. curry/uncurry (−xA ⊣ A→−)"
-- curry :: ((a,b)->c) -> a->b->c  (rightAdjunct)
-- uncurry :: (a->b->c) -> (a,b)->c  (leftAdjunct)
let addPair :: (Int, Int) -> Int; addPair (x,y) = x + y
let addCurried :: Int -> Int -> Int; addCurried = curry addPair
-- round-trip: curry . uncurry = id
print $ curry addPair 3 4        -- 7 = addCurried 3 4
print $ uncurry addCurried (3,4) -- 7 = addPair (3,4)
print $ all (\(x,y) -> curry (uncurry addCurried) x y == addCurried x y) [(x,y) | x <- [1..5::Int], y <- [1..5]]  -- True

putSection "2. (,) ⊣ (→) via fanout/apply"
let fanout' f g x = (f x, g x)    -- Δ → (a,b)
let apply' (f,x) = f x             -- (a→b, a) → b
print $ fanout' (+1) (*2) 5        -- (6, 10)
print $ apply' ((+1), 5)           -- 6

putSection "3. Free ⊣ Forget"
-- Список как свободный моноид
let freeMonoid xs = xs             -- включение в свободный
let forget f xs = foldMap f xs     -- единственный гомоморфизм
-- Закон: forget id = fold
print $ forget (:[]) [1,2,3 :: Int]     -- [1,2,3]
print $ forget show [1,2,3 :: Int]      -- ["1","2","3"]

putSection "4. State ⊣ Store (−xS ⊣ S→−)"
let runState'' (State' f) s = f s
-- Монада State: return и bind
let ret x = State' $ \s -> (x, s)
let bind (State' m) f = State' $ \s -> let (a,s') = m s in runState' (f a) s'
let prog = ret 42 `bind` (\x -> ret (x * 2))
print $ runState'' prog "unused"   -- (84, "unused")

## Диаграмма: структура сопряжения

Визуальная схема сопряжения F ⊣ G.

![Сопряжение F ⊣ G](../diagrams/adjunctions/adj_adjunction.svg)


---
**← Предыдущий:** [YonedaLemma](YonedaLemma.ipynb)  |  **Следующий →** [KanExtensions](KanExtensions.ipynb)